---
title: "FHIR Graph RAG with Deterministic Parsing and AWS Bedrock"
format: html
jupyter: python3
---

## Introduction

Traditional RAG retrieves chunks of text ranked by vector similarity. For clinical records, this loses structure: a query about a patient's medications during a specific encounter requires knowing _which_ observations and prescriptions are linked - relationships that live in the graph, not in prose proximity.

**Graph RAG** replaces (or augments) the flat vector store with a property graph. Retrieval becomes a graph traversal: find the encounter, walk to its conditions, prescriptions, and lab values. This preserves the relational structure of FHIR natively.

This notebook uses a **deterministic parser** to build the graph - directly mapping FHIR R4 JSON fields to Neo4j nodes and relationships with no LLM calls. Once the graph is loaded, `GraphCypherQAChain` uses the Bedrock model to translate natural-language questions into Cypher queries, execute them against Neo4j, and return a grounded answer.

---

## Setup

In [1]:
import json
from collections import defaultdict

from neo4j import GraphDatabase
from langchain_aws import ChatBedrockConverse
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain

NEO4J_URI      = "bolt://localhost:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "password"

FHIR_FILE = "synthea-fhir/Addie421_Isabelle619_Johns824_c72a8c6e-1dfd-0a8b-20e7-066eadd4931c.json"
MODEL_ID  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

AWS credentials are read from `~/.aws/credentials` or environment variables (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_REGION`). No credentials are hard-coded here.

In [2]:
llm = ChatBedrockConverse(
    model_id=MODEL_ID,
)

---

## Load FHIR Bundle

In [3]:
with open(FHIR_FILE) as f:
    bundle = json.load(f)

entries = bundle.get("entry", [])

type_counts = defaultdict(int)
for entry in entries:
    rt = entry.get("resource", {}).get("resourceType", "Unknown")
    type_counts[rt] += 1

print(f"Total entries: {len(entries)}\n")
print(f"{'Resource Type':<25} {'Count':>5}")
print("-" * 32)
for rt, count in sorted(type_counts.items()):
    print(f"{rt:<25} {count:>5}")

Total entries: 476

Resource Type             Count
--------------------------------
CarePlan                      1
CareTeam                      1
Claim                        48
Condition                    27
Device                        4
DiagnosticReport             52
DocumentReference            36
Encounter                    36
ExplanationOfBenefit         48
ImagingStudy                  2
Immunization                  9
Medication                    6
MedicationAdministration      6
MedicationRequest            12
Observation                  83
Patient                       1
Procedure                    87
Provenance                    1
SupplyDelivery               16


---

## Deterministic Graph Building

Each FHIR resource type is handled by a dedicated function that extracts a fixed set of properties and FHIR reference relationships. The results are written directly to Neo4j using Cypher `MERGE` - idempotent, so re-running a cell won't create duplicate nodes.

### Helper Functions

In [4]:
def ref_id(reference_string):
    """Return the bare UUID from a FHIR reference.

    "urn:uuid:abc-123"  ->  "abc-123"
    "Patient/abc-123"   ->  "abc-123"
    """
    if not reference_string:
        return None
    if reference_string.startswith("urn:uuid:"):
        return reference_string[len("urn:uuid:"):]
    if "/" in reference_string:
        return reference_string.split("/")[-1]
    return reference_string


def get_ref(resource, *keys):
    """Safely walk a chain of dict keys and call ref_id on the final 'reference' value."""
    obj = resource
    for key in keys:
        if not isinstance(obj, dict):
            return None
        obj = obj.get(key)
        if obj is None:
            return None
    if isinstance(obj, dict):
        return ref_id(obj.get("reference"))
    return None

### Resource Handlers

In [5]:
def handle_patient(r):
    name_list = r.get("name") or []
    first = name_list[0] if name_list else {}
    given = " ".join(first.get("given") or []) or None
    family = first.get("family")
    full_name = f"{given} {family}" if given and family else None

    addr_list = r.get("address") or []
    addr = addr_list[0] if addr_list else {}

    telecom = r.get("telecom") or []
    phone = telecom[0].get("value") if telecom else None

    props = {
        "id": r.get("id"),
        "name": full_name,
        "gender": r.get("gender"),
        "birthDate": r.get("birthDate"),
        "city": addr.get("city"),
        "state": addr.get("state"),
        "phone": phone,
    }
    return "Patient", props, []


def handle_encounter(r):
    # Synthea R4 emits "class" as a single Coding object; R4B changed it to a list.
    class_field = r.get("class") or {}
    if isinstance(class_field, list):
        class_code = class_field[0].get("code") if class_field else None
    else:
        class_code = class_field.get("code")

    type_list = r.get("type") or []
    encounter_type = type_list[0].get("text") if type_list else None

    period = r.get("period") or {}
    props = {
        "id": r.get("id"),
        "status": r.get("status"),
        "class": class_code,
        "type": encounter_type,
        "start": period.get("start"),
        "end": period.get("end"),
    }
    refs = [
        ("FOR_PATIENT", "Patient", get_ref(r, "subject")),
    ]
    return "Encounter", props, refs


def handle_condition(r):
    clinical_status = r.get("clinicalStatus") or {}
    codings = clinical_status.get("coding") or []
    status_code = codings[0].get("code") if codings else None

    props = {
        "id": r.get("id"),
        "display": (r.get("code") or {}).get("text"),
        "onset": r.get("onsetDateTime"),
        "clinicalStatus": status_code,
    }
    refs = [
        ("FOR_PATIENT", "Patient", get_ref(r, "subject")),
        ("DURING_ENCOUNTER", "Encounter", get_ref(r, "encounter")),
    ]
    return "Condition", props, refs


def handle_observation(r):
    vq  = r.get("valueQuantity")
    vcc = r.get("valueCodeableConcept")
    vs  = r.get("valueString")

    if vq:
        value = str(vq.get("value"))
        unit  = vq.get("unit")
    elif vcc:
        value = vcc.get("text")
        unit  = None
    elif vs:
        value = vs
        unit  = None
    else:
        value = None
        unit  = None

    props = {
        "id":      r.get("id"),
        "display": (r.get("code") or {}).get("text"),
        "date":    r.get("effectiveDateTime"),
        "status":  r.get("status"),
        "value":   value,
        "unit":    unit,
    }
    refs = [
        ("FOR_PATIENT",      "Patient",   get_ref(r, "subject")),
        ("DURING_ENCOUNTER", "Encounter", get_ref(r, "encounter")),
    ]
    return "Observation", props, refs


def handle_procedure(r):
    pp    = r.get("performedPeriod") or {}
    start = pp.get("start") or r.get("performedDateTime")

    props = {
        "id":      r.get("id"),
        "display": (r.get("code") or {}).get("text"),
        "status":  r.get("status"),
        "start":   start,
    }
    refs = [
        ("FOR_PATIENT",      "Patient",   get_ref(r, "subject")),
        ("DURING_ENCOUNTER", "Encounter", get_ref(r, "encounter")),
    ]
    return "Procedure", props, refs


def handle_medication(r):
    props = {
        "id":      r.get("id"),
        "display": (r.get("code") or {}).get("text"),
        "status":  r.get("status"),
    }
    return "Medication", props, []


def handle_medication_request(r):
    props = {
        "id":         r.get("id"),
        "status":     r.get("status"),
        "intent":     r.get("intent"),
        "authoredOn": r.get("authoredOn"),
    }
    refs = [
        ("FOR_PATIENT",           "Patient",   get_ref(r, "subject")),
        ("DURING_ENCOUNTER",      "Encounter", get_ref(r, "encounter")),
        ("PRESCRIBED_MEDICATION", "Medication", get_ref(r, "medicationReference")),
    ]
    return "MedicationRequest", props, refs


def handle_immunization(r):
    props = {
        "id":      r.get("id"),
        "vaccine": (r.get("vaccineCode") or {}).get("text"),
        "date":    r.get("occurrenceDateTime"),
        "status":  r.get("status"),
    }
    refs = [
        # Immunization uses .patient, not .subject (per FHIR R4 spec)
        ("FOR_PATIENT",      "Patient",   get_ref(r, "patient")),
        ("DURING_ENCOUNTER", "Encounter", get_ref(r, "encounter")),
    ]
    return "Immunization", props, refs


RESOURCE_HANDLERS = {
    "Patient":           handle_patient,
    "Encounter":         handle_encounter,
    "Condition":         handle_condition,
    "Observation":       handle_observation,
    "Procedure":         handle_procedure,
    "Medication":        handle_medication,
    "MedicationRequest": handle_medication_request,
    "Immunization":      handle_immunization,
}

### Neo4j Write Helpers

In [6]:
def write_node(session, label, props):
    set_clause = ", ".join(f"n.{k} = ${k}" for k in props if k != "id")
    query = f"MERGE (n:{label} {{id: $id}}) SET {set_clause}"
    session.run(query, **props)


def write_rel(session, src_label, src_id, rel_type, tgt_label, tgt_id):
    query = (
        f"MATCH (src:{src_label} {{id: $src_id}}) "
        f"MATCH (tgt:{tgt_label} {{id: $tgt_id}}) "
        f"MERGE (src)-[:{rel_type}]->(tgt)"
    )
    session.run(query, src_id=src_id, tgt_id=tgt_id)

### Load into Neo4j

In [7]:
counts = {}

with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as driver:
    driver.verify_connectivity()
    print(f"Connected to {NEO4J_URI}")

    with driver.session() as session:
        for entry in entries:
            resource      = entry.get("resource", {})
            resource_type = resource.get("resourceType")
            handler       = RESOURCE_HANDLERS.get(resource_type)
            if handler is None:
                continue

            label, props, refs = handler(resource)
            write_node(session, label, props)

            for rel_type, tgt_label, tgt_id in refs:
                if tgt_id:
                    write_rel(session, label, props["id"], rel_type, tgt_label, tgt_id)

            counts[label] = counts.get(label, 0) + 1

print("\nNodes written:")
for label, count in sorted(counts.items()):
    print(f"  {label}: {count}")

Connected to bolt://localhost:7687

Nodes written:
  Condition: 27
  Encounter: 36
  Immunization: 9
  Medication: 6
  MedicationRequest: 12
  Observation: 83
  Patient: 1
  Procedure: 87


### Graph Summary

In [8]:
with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as driver:
    with driver.session() as session:
        result = session.run(
            "MATCH (n) RETURN labels(n)[0] AS label, count(*) AS count ORDER BY label"
        )
        print("Nodes:")
        for record in result:
            print(f"  {record['label']}: {record['count']}")

        result = session.run(
            "MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS count ORDER BY rel"
        )
        print("\nRelationships:")
        for record in result:
            print(f"  {record['rel']}: {record['count']}")

Nodes:
  Condition: 27
  Encounter: 36
  Immunization: 9
  Medication: 6
  MedicationRequest: 12
  Observation: 83
  Patient: 1
  Procedure: 87

Relationships:
  DURING_ENCOUNTER: 218
  FOR_PATIENT: 254
  PRESCRIBED_MEDICATION: 6


---

## Q&A with GraphCypherQAChain

`GraphCypherQAChain` works in two steps: the LLM generates a Cypher query from the user's question (using the live graph schema for grounding), Neo4j executes it, and the LLM synthesizes a natural-language answer from the results. Every answer is anchored to what the query actually returns - no hallucinated clinical facts.

In [9]:
neo4j_graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USER,
    password=NEO4J_PASSWORD,
)

In [10]:
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=neo4j_graph,
    verbose=True,
    allow_dangerous_requests=True,
)

### Example Queries

In [11]:
result = chain.invoke({"query": "What conditions does the patient have?"})
print(result["result"])



> Entering new GraphCypherQAChain chain...


Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `clinicalStatus` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=21, offset=67>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 67, 'line': 2, 'column': 21}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (p:Patient)-[:FOR_PATIENT]-(c:Condition)\nRETURN c.display, c:clinicalStatus, c.onset'


Generated Cypher:
MATCH (p:Patient)-[:FOR_PATIENT]-(c:Condition)
RETURN c.display, c:clinicalStatus, c.onset
Full Context:
[{'c.display': 'Received higher education (finding)', 'c:clinicalStatus': False, 'c.onset': '1998-10-06T09:16:02+00:00'}, {'c.display': 'Loss of teeth (disorder)', 'c:clinicalStatus': False, 'c.onset': '2002-11-05T08:17:36+00:00'}, {'c.display': 'Full-time employment (finding)', 'c:clinicalStatus': False, 'c.onset': '2008-10-21T08:54:30+00:00'}, {'c.display': 'Stress (finding)', 'c:clinicalStatus': False, 'c.onset': '2011-03-29T08:55:47+00:00'}, {'c.display': 'Body mass index 30+ - obesity (finding)', 'c:clinicalStatus': False, 'c.onset': '2014-10-28T08:17:36+00:00'}, {'c.display': 'Normal pregnancy (finding)', 'c:clinicalStatus': False, 'c.onset': '2017-02-28T08:17:36+00:00'}, {'c.display': 'Miscarriage in first trimester (disorder)', 'c:clinicalStatus': False, 'c.onset': '2017-02-28T08:17:36+00:00'}, {'c.display': 'Blighted ovum (disorder)', 'c:clinicalStatus': F

In [12]:
result = chain.invoke({"query": "What medications were prescribed and during which encounters?"})
print(result["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (mr:MedicationRequest)-[:PRESCRIBED_MEDICATION]->(m:Medication)
MATCH (mr)-[:DURING_ENCOUNTER]->(e:Encounter)
RETURN m.display AS medication, e.id AS encounter_id, e.start AS encounter_start, e.end AS encounter_end

Full Context:
[{'medication': 'Etonogestrel 68 MG Drug Implant', 'encounter_id': 'c72a8c6e-1dfd-0a8b-8f3c-7477c46b84fd', 'encounter_start': '2017-03-28T08:17:36+00:00', 'encounter_end': '2017-03-28T08:48:56+00:00'}, {'medication': 'sodium fluoride 0.0272 MG/MG Oral Gel', 'encounter_id': 'c72a8c6e-1dfd-0a8b-ac4a-7f8b12447ab4', 'encounter_start': '2017-11-14T08:17:36+00:00', 'encounter_end': '2017-11-14T11:54:01+00:00'}, {'medication': 'levonorgestrel 0.000813 MG/HR Intrauterine System [Liletta]', 'encounter_id': 'c72a8c6e-1dfd-0a8b-b3d9-11546d5d4f37', 'encounter_start': '2018-03-23T08:48:56+00:00', 'encounter_end': '2018-03-23T09:03:56+00:00'}, {'medication': 'sodium fluoride 0.0272 MG/MG Oral Gel', 'encoun

In [13]:
result = chain.invoke({"query": "How many observations were recorded for this patient?"})
print(result["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Patient)-[:FOR_PATIENT]-(o:Observation)
RETURN COUNT(o) as observation_count

Full Context:
[{'observation_count': 83}]

> Finished chain.
83 observations were recorded for this patient.


In [14]:
result = chain.invoke({"query": "What immunizations has the patient received and when?"})
print(result["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Patient)-[:FOR_PATIENT]-(i:Immunization)
RETURN p.name, i.vaccine, i.date
ORDER BY i.date
Full Context:
[{'p.name': 'Addie421 Isabelle619 Johns824', 'i.vaccine': 'Influenza, split virus, trivalent, PF', 'i.date': '2017-10-31T08:17:36+00:00'}, {'p.name': 'Addie421 Isabelle619 Johns824', 'i.vaccine': 'Influenza, split virus, trivalent, PF', 'i.date': '2020-08-18T08:17:36+00:00'}, {'p.name': 'Addie421 Isabelle619 Johns824', 'i.vaccine': 'Hep A, adult', 'i.date': '2020-08-18T08:17:36+00:00'}, {'p.name': 'Addie421 Isabelle619 Johns824', 'i.vaccine': 'COVID-19, mRNA, LNP-S, PF, 100 mcg/0.5mL dose or 50 mcg/0.25mL dose', 'i.date': '2021-01-19T08:17:36+00:00'}, {'p.name': 'Addie421 Isabelle619 Johns824', 'i.vaccine': 'COVID-19, mRNA, LNP-S, PF, 100 mcg/0.5mL dose or 50 mcg/0.25mL dose', 'i.date': '2021-02-16T08:17:36+00:00'}, {'p.name': 'Addie421 Isabelle619 Johns824', 'i.vaccine': 'Influenza, split virus, trivalent, PF', 